<a href="https://colab.research.google.com/github/Arfa-Tariq/learning-ai-engineering/blob/main/projects/05-UnivesityHelpdeskBot/university_helpdesk_bot.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# University Helpdesk Bot – System Overview
## What It Does
- A student asks a question → The bot:

- Classifies the question type (admissions, courses, faculty, campus, etc.)

- Checks safety (moderation) – filters harmful content

- Retrieves relevant information from a knowledge base

- Evaluates the response quality

- Reasons if follow-up is needed

- Returns a helpful, safe answer

## Architecture

[Student asks question]  
        ↓  
[Moderation Check] → If unsafe: Block + Alert  
         ↓  
[Intent Classification] → Category (admissions / courses / financial / campus / other)  
         ↓  
[Information Retrieval] → Search knowledge base  
         ↓  
[Response Generation] → Formatted, helpful answer  
         ↓  
[Evaluation] → Check relevance, completeness, safety  
         ↓  
[Reasoning Pipeline] → Determine if follow-up needed  
         ↓  
[Response to Student]

In [118]:
#Basic Setup
!pip install -q groq gradio

from groq import Groq
from google.colab import userdata
import json
import gradio as gr
import os
import ast

In [119]:
# Get API key from Colab secrets
GROQ_API_KEY = userdata.get('groq_key')
client = Groq(api_key=GROQ_API_KEY)

In [121]:
def get_completion_from_messages(messages,
                                 model="llama-3.1-8b-instant",
                                 temperature=0,):
    response = client.chat.completions.create(
        model=model,
        messages=messages,
        temperature=temperature, # this is the degree of randomness of the model's output
    )
    return response.choices[0].message.content


In [120]:
with open("knowledge_base.json", "r") as f:
    kb = json.load(f)

In [99]:
def determine_follow_up(question, response):
    """
    Placeholder for follow-up suggestion logic.
    """
    return None

## STEP 1: Moderation of User input

In [125]:
def moderation_check(question):
    """
    Returns (is_safe: bool, category: str).
    """
    if not question or not question.strip():
        return False, "empty"

    messages = [
        {
            "role": "system",
            "content": (
                "You are a strict content moderator for a university helpdesk.\n"
                "Classify the user input as EXACTLY one of: safe / harmful / illegal / adult\n\n"
                "Examples:\n"
                "- 'When is the FAFSA deadline?' → safe\n"
                "- 'I want to blow up the library' → harmful\n"
                "- 'Where can I buy cocaine?' → illegal\n"
                "- 'How to hack the student portal?' → illegal\n"
                "- 'Can you help me cheat?' → illegal\n"
                "- 'Tell me an explicit joke' → adult\n\n"
                "Return ONLY ONE WORD: safe / harmful / illegal / adult"
            ),
        },
        {"role": "user", "content": f"Input: {question}"},
    ]

    raw = get_completion_from_messages(messages, temperature=0)
    category = raw.strip().lower().split()[0]   # take first word only

    valid = {"safe", "harmful", "illegal", "adult"}
    if category not in valid:
        category = "unsafe"

    return category == "safe", category


In [126]:
test_questions = [
    "When is the FAFSA deadline?",
    "I hate this university and want to destroy it",
    "How to buy illegal drugs?",
    "How to hack the professor's computer?",
    "What courses are required for BS CIS?",
    "Can you help me cheat on my exam?",
    "Tell me about the hackathon"
]

print("🧪 Testing Moderation\n")
print("=" * 60)

for q in test_questions:
    is_safe, categories = moderation_check(q)
    status = "✅ PASS" if is_safe else "❌ BLOCKED"
    print(f"{status} | Categories: {categories} | Question: {q[:50]}...")

🧪 Testing Moderation

✅ PASS | Categories: safe | Question: When is the FAFSA deadline?...
❌ BLOCKED | Categories: harmful | Question: I hate this university and want to destroy it...
❌ BLOCKED | Categories: illegal | Question: How to buy illegal drugs?...
❌ BLOCKED | Categories: illegal | Question: How to hack the professor's computer?...
✅ PASS | Categories: safe | Question: What courses are required for BS CIS?...
❌ BLOCKED | Categories: illegal | Question: Can you help me cheat on my exam?...
✅ PASS | Categories: safe | Question: Tell me about the hackathon...


## Step2: Classify intent of the user query

In [127]:
def classify_intent(question):
    """
    Asks the LLM to return a JSON list of {category, specific_query?} objects.
    """
    system_message = """
You will be provided with a university helpdesk query delimited by ####.
Return a valid JSON array of objects. Each object must have:
  - "category": one of the allowed categories below
  - "specific_query" (optional): one of the allowed keys listed under that category

Allowed categories and their specific_query keys:

university        → (no specific_query needed)
department        → (no specific_query needed)
location          → city | country
contact           → phone | email | website | portal
admissions        → bs | ms | phd
programs          → bs_core_courses | bs_credits | ms_credits | phd_credits
faculty           → dr_ahmed_ali | dr_sara_khan | dr_hassan_raza | dr_umar_malik
labs              → ai_lab | networks_lab | software_lab
key_dates         → career_fair | hackathon | tech_talks
bs_graduation_requirements → credits | min_cgpa | internship | capstone

Rules:
- Use snake_case for specific_query values (e.g. "dr_ahmed_ali", "ai_lab", "career_fair").
- If the query is broad (e.g. "tell me about admissions"), omit specific_query.
- If multiple topics are mentioned, include one object per topic.
- If nothing matches, return [].
- Output ONLY the JSON array, no explanation, no markdown fences.

Example outputs:
[{"category": "admissions", "specific_query": "bs"}]
[{"category": "faculty", "specific_query": "dr_ahmed_ali"}, {"category": "admissions"}]
[]
"""

    messages = [
        {"role": "system", "content": system_message},
        {"role": "user", "content": f"####{question}####"},
    ]
    return get_completion_from_messages(messages)

In [128]:
def parse_classification(raw_output):
    """
    Safely parses the LLM's JSON string into a Python list.
    """
    if not raw_output:
        return []
    if isinstance(raw_output, list):
        return raw_output

    text = raw_output.strip()
    # Strip markdown code fences if present
    if text.startswith("```"):
        lines = text.splitlines()
        text = "\n".join(lines[1:-1] if lines[-1].strip() == "```" else lines[1:])

    try:
        parsed = json.loads(text)
    except json.JSONDecodeError:
        try:
            parsed = ast.literal_eval(text)
        except (ValueError, SyntaxError):
            return []

    if isinstance(parsed, list):
        return parsed
    if isinstance(parsed, dict):
        return [parsed]
    return []

#Step 3: KB retrieval

In [129]:
def get_kb_entry(knowledge_base, category, specific_query=None):
    """
    Retrieves data from the KB. specific_query must already be a valid KB key
    (snake_case). Falls back to returning the whole category if key not found.
    """
    if category not in knowledge_base:
        return None

    if specific_query is None:
        return knowledge_base[category]

    data = knowledge_base[category]
    if not isinstance(data, dict):
        return data  # e.g. category value is a plain string

    # Exact key match
    if specific_query in data:
        return data[specific_query]

    # Case-insensitive fallback
    sq_lower = specific_query.lower()
    for key in data:
        if key.lower() == sq_lower:
            return data[key]

    # Partial match (last resort)
    for key in data:
        if sq_lower in key.lower() or key.lower() in sq_lower:
            return data[key]

    return None  # not found → caller shows "no info" message

In [130]:
def format_data(data, indent=2):
    """Recursively formats KB data into readable text."""
    pad = " " * indent
    lines = []
    if isinstance(data, dict):
        for k, v in data.items():
            key_label = k.replace("_", " ").title()
            if isinstance(v, (dict, list)):
                lines.append(f"{pad}{key_label}:")
                lines.append(format_data(v, indent + 2))
            else:
                lines.append(f"{pad}{key_label}: {v}")
    elif isinstance(data, list):
        for item in data:
            lines.append(f"{pad}• {item}")
    else:
        lines.append(f"{pad}{data}")
    return "\n".join(lines)

In [131]:
def generate_response_from_kb(knowledge_base, classification_list):
    """
    Builds a human-readable answer from a list of {category, specific_query?} items.
    """
    if not classification_list:
        return "I couldn't find any relevant information. Please try rephrasing your question."

    output_parts = []

    for item in classification_list:
        category = item.get("category", "").strip()
        specific_query = item.get("specific_query", "").strip() or None

        if not category:
            continue

        data = get_kb_entry(knowledge_base, category, specific_query)

        if data is None:
            label = specific_query or category
            output_parts.append(f"I don't have specific information about '{label}'.")
            continue

        # Build a readable heading
        if specific_query:
            heading = f"{category.replace('_', ' ').title()} › {specific_query.replace('_', ' ').title()}"
        else:
            heading = category.replace("_", " ").title()

        output_parts.append(f"**{heading}**")
        output_parts.append(format_data(data))
        output_parts.append("")  # blank line between sections

    return "\n".join(output_parts).strip() or "No relevant information found."

## Main Processing Function


In [134]:
def process_user_message(user_input, all_messages, debug=True):
    """
    7-step helpdesk pipeline:
      1. Moderate input
      2. Classify intent → category + specific_query
      3. Retrieve KB information
      4. Generate a friendly LLM response using KB data
      5. Moderate output
      6. Evaluate whether the response actually answers the question
      7. Return final response or escalate to human

    Returns: (response_str, updated_all_messages)
    """
    delimiter = "```"

    # ── Step 1: Moderate user input ───────────────────────────────────────────
    is_safe, mod_category = moderation_check(user_input)
    if not is_safe:
        if debug:
            print("Step 1: Input flagged by Moderation.")
        if mod_category == "empty":
            return "Please enter a question so I can help you.", all_messages
        return "Sorry, we cannot process this request.", all_messages
    if debug:
        print("Step 1: Input passed moderation check.")

    # ── Step 2: Classify intent ───────────────────────────────────────────────
    raw_intent = classify_intent(user_input)
    classification = parse_classification(raw_intent)
    if debug:
        print(f"Step 2: Extracted intent: {classification}")

    if not classification:
        return (
            "I'm not sure what you're asking about. "
            "You can ask me about admissions, programs, faculty, labs, key dates, or contact info.",
            all_messages,
        )

    # ── Step 3: Retrieve KB information ──────────────────────────────────────
    kb_information = generate_response_from_kb(kb, classification)
    if debug:
        print("Step 3: Looked up information from knowledge base.")

    # ── Step 4: Generate a friendly LLM response ──────────────────────────────
    # KB data goes in the system message as reference material — NOT as a fake
    # assistant turn. Putting it in the assistant role causes the model to treat
    # it as something it already said and repeat every field in subsequent turns.
    system_message = (
        "You are a helpful and friendly student advisor for the CIS Department at NED University. "
        "Answer the student's question using ONLY the reference information provided below. "
        "Be concise and warm. End with one relevant follow-up question.\n\n"
        f"Reference information:\n{kb_information}"
    )

    messages = [
        {"role": "system", "content": system_message},
        {"role": "user", "content": f"{delimiter}{user_input}{delimiter}"},
    ]

    final_response = get_completion_from_messages(all_messages + messages)
    if debug:
        print("Step 4: Generated response to user question.")

    # Append only the real user question and real final response to history —
    # never the raw KB dump, so it doesn't bleed into future turns.
    all_messages = all_messages + [
        {"role": "user", "content": user_input},
        {"role": "assistant", "content": final_response},
    ]

    # ── Step 5: Moderate the output ───────────────────────────────────────────
    is_output_safe, output_mod_category = moderation_check(final_response)
    if not is_output_safe:
        if debug:
            print(f"Step 5: Response flagged by Moderation ({output_mod_category}).")
        return "Sorry, we cannot provide this information.", all_messages
    if debug:
        print("Step 5: Response passed moderation check.")

    # ── Step 6: Evaluate whether the response answers the question ────────────
    eval_system = (
        "You are a quality evaluator for a university helpdesk. "
        "You will be given the student's question, the knowledge base information that was available, "
        "and the agent's response. "
        "Judge ONLY whether the agent used the available information to address the question. "
        "Do NOT penalise the agent for information that was not in the knowledge base. "
        "Reply with exactly 'Y' if the response is relevant and helpful given what was available, "
        "or 'N' if it is completely off-topic or ignores the question entirely."
    )
    eval_user = (
        f"Student question: {delimiter}{user_input}{delimiter}\n\n"
        f"Available knowledge base information: {delimiter}{kb_information}{delimiter}\n\n"
        f"Agent response: {delimiter}{final_response}{delimiter}\n\n"
        "Does the response sufficiently answer the question given the available information? "
        "Answer with Y or N only."
    )
    eval_messages = [
        {"role": "system", "content": eval_system},
        {"role": "user", "content": eval_user},
    ]
    evaluation_response = get_completion_from_messages(eval_messages)
    if debug:
        print(f"Step 6: Model evaluated the response → '{evaluation_response.strip()}'")

    # ── Step 7: Approve or escalate ───────────────────────────────────────────
    if "Y" in evaluation_response.upper():
        if debug:
            print("Step 7: Model approved the response.")
        return final_response, all_messages
    else:
        if debug:
            print("Step 7: Model disapproved the response.")
        fallback = (
            "I'm unable to provide a complete answer to your question right now. "
            "Please reach out to the CIS department directly:\n"
            "📧 cis@neduet.edu.pk  |  📞 +92-21-12345678"
        )
        return fallback, all_messages


In [135]:
# Gradio UI
# ══════════════════════════════════════════════════════════════════════════════

def chat_interface(message, history, debug):
    """Gradio ChatInterface handler. Rebuilds all_messages from Gradio history."""
    all_messages = []
    for user_msg, bot_msg in history:
        all_messages.append({"role": "user", "content": user_msg})
        if bot_msg:
            all_messages.append({"role": "assistant", "content": bot_msg})

    response, _ = process_user_message(message, all_messages, debug=debug)
    return response


demo = gr.ChatInterface(
    fn=chat_interface,
    title="NED CIS Department Helpdesk",
    description="Ask anything about admissions, programs, faculty, labs, key dates, and more.",
    theme="soft",
    additional_inputs=[
        gr.Checkbox(label="Show Debug Info in Console", value=False)
    ],
)

if __name__ == "__main__":
    # Quick smoke test
    print("Running quick pipeline test...\n")
    test_input = "Tell me about BS admissions and Dr. Ahmed Ali"
    response, _ = process_user_message(test_input, [], debug=True)
    print(f"\nFinal response:\n{response}\n")

    demo.launch(share=False)

/usr/local/lib/python3.12/dist-packages/gradio/chat_interface.py:347: UserWarning: The 'tuples' format for chatbot messages is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style 'role' and 'content' keys.
  self.chatbot = Chatbot(


Running quick pipeline test...

Step 1: Input passed moderation check.
Step 2: Extracted intent: [{'category': 'admissions', 'specific_query': 'bs'}, {'category': 'faculty', 'specific_query': 'dr_ahmed_ali'}]
Step 3: Looked up information from knowledge base.
Step 4: Generated response to user question.
Step 5: Response passed moderation check.
Step 6: Model evaluated the response → 'Y'
Step 7: Model approved the response.

Final response:
Welcome to NED University's CIS Department. I'd be happy to help you with your questions.

For BS admissions, the duration of the program is 4 years. To be eligible, you'll need to have completed FSc or A-Levels with at least 60% marks. Additionally, you'll need to take the NED Entry Test, which is mandatory for admission.

As for deadlines, please note that the Fall semester deadline is August 1, and the Spring semester deadline is December 15.

Regarding tuition fees, it's PKR 95,000 per semester.

We also offer scholarships to our students, which 

<IPython.core.display.Javascript object>